# 09 · Business aviation performance analysis

This notebook converts the EUROCONTROL flight records into a business-facing
view of network demand, punctuality, severity and operational recovery.

**Scope**

- Scheduled commercial traffic only: `ICAO Flight Type == 'S'`.
- Directional routes: `ADEP → ADES`.
- Development periods through December 2022.
- March 2023 is excluded and remains the blind model test.
- Counts refer to operated flights, not passengers, seats or revenue.

The notebook adds a new analysis layer. It does not replace notebooks 01–03.

In [1]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.business_eda import (
    BusinessAnalysisConfig,
    airport_performance,
    business_hypothesis_tests,
    development_flight_paths,
    executive_route_views,
    export_business_analysis,
    hypothesis_test_catalog,
    load_dimension_labels,
    numeric_correlation_table,
    operator_performance,
    overall_kpis,
    plot_departure_arrival_recovery,
    plot_airport_reliability_rankings,
    plot_airport_volume_reliability,
    plot_route_volume_reliability,
    plot_statistical_method_explainer,
    plot_time_reliability_heatmap,
    plot_top_route_comparison,
    read_business_flights,
    route_performance,
    route_threshold_sensitivity,
    scan_route_volume,
)

RAW_FLIGHTS = PROJECT_ROOT / "data" / "raw" / "flights"
RAW_ICAO = PROJECT_ROOT / "data" / "raw" / "icao"
BASE_OUTPUT_ROOT = PROJECT_ROOT / "reports" / "business_eda"
all_flight_files = sorted(RAW_FLIGHTS.glob("Flights_*.csv.gz"))
assert all_flight_files, "No flight files were found"

## 1. Reproducible business rules

The configuration below makes every reporting choice visible. The executive
route ranking requires at least 500 operated flights and activity in three
observed periods. A broader 100-flight view is retained for network coverage.

Set `RUN_FULL_ANALYSIS = True` when producing the final report. The default
smoke mode reads only a few rows and is safe for rapid validation.

In [2]:
CONFIG = BusinessAnalysisConfig(
    test_start="2023-01-01",
    executive_min_route_flights=500,
    executive_min_route_periods=3,
    executive_min_operator_flights=1_000,
    route_plot_min_flights=2,
    airport_plot_min_flights=30,
)

RUN_FULL_ANALYSIS = False
SMOKE_ROWS_PER_FILE = 2_000
MAX_ROWS_PER_FILE = None if RUN_FULL_ANALYSIS else SMOKE_ROWS_PER_FILE
OUTPUT_ROOT = (
    BASE_OUTPUT_ROOT
    if RUN_FULL_ANALYSIS
    else PROJECT_ROOT / "reports" / "business_eda_smoke"
)

print({
    "run_full_analysis": RUN_FULL_ANALYSIS,
    "max_rows_per_file": MAX_ROWS_PER_FILE,
    "blind_test_start": CONFIG.test_start,
    "output_root": str(OUTPUT_ROOT),
})

# Exclude the blind-test file before any CSV reader opens it.
flight_files = development_flight_paths(all_flight_files, CONFIG)
print({
    "development_files": [path.name for path in flight_files],
    "excluded_files": sorted(set(path.name for path in all_flight_files) - set(path.name for path in flight_files)),
})

{'run_full_analysis': False, 'max_rows_per_file': 2000, 'blind_test_start': '2023-01-01', 'output_root': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\business_eda_smoke'}
{'development_files': ['Flights_20211201_20211231.csv.gz', 'Flights_20220301_20220331.csv.gz', 'Flights_20220601_20220630.csv.gz', 'Flights_20220901_20220930.csv.gz', 'Flights_20221201_20221231.csv.gz'], 'excluded_files': ['Flights_20230301_20230331.csv.gz']}


## 2. Is a 500-flight route threshold sufficiently inclusive?

This lightweight scan reads only route, period and flight-type columns. It uses
all development files even when the rest of the notebook runs in smoke mode.

Two thresholds are useful for different questions:

- **500 flights + 3 periods:** defensible executive comparison.
- **100 flights + 3 periods:** broader network monitoring and discovery.

In [3]:
# Exact volume scan; March 2023 is rejected by the shared configuration.
volume_started = time.perf_counter()
route_volume = scan_route_volume(flight_files, CONFIG)
volume_sensitivity = route_threshold_sensitivity(route_volume, CONFIG)
display(volume_sensitivity.round(2))
print(f"Route-volume scan: {time.perf_counter() - volume_started:.1f} seconds")

,minimum_flights,minimum_periods,eligible_routes,route_coverage_pct,covered_flights,flight_coverage_pct
0,100,1,7491,25.73,2507818,82.25
1,100,2,7480,25.69,2506504,82.21
2,100,3,7417,25.48,2498340,81.94
3,250,1,3291,11.30,1850812,60.70
4,250,2,3291,11.30,1850812,60.70
5,250,3,3289,11.30,1850230,60.68
6,500,1,1254,4.31,1138255,37.33
7,500,2,1254,4.31,1138255,37.33
8,500,3,1254,4.31,1138255,37.33
9,1000,1,331,1.14,502239,16.47


Route-volume scan: 16.0 seconds


### How to interpret the threshold table

`eligible_routes` measures breadth; `flight_coverage_pct` measures how much of
the operated network remains. A high minimum volume increases statistical
stability but removes thin routes. The final report therefore presents both the
executive and broad-coverage views rather than hiding this trade-off.

## 3. Build the compact analytical flight table

The raw compressed files are read in chunks. Only report variables are kept,
continuous values are stored as 32-bit floats, and repeated text fields become
categories. This keeps the full analysis feasible on a low-memory computer.

In [4]:
load_started = time.perf_counter()
flights = read_business_flights(
    flight_files,
    CONFIG,
    chunksize=100_000,
    max_rows_per_file=MAX_ROWS_PER_FILE,
)

# This assertion protects the blind model test.
assert flights["FILED OFF BLOCK TIME"].max() < pd.Timestamp(CONFIG.test_start)
print({
    "analysis_rows": len(flights),
    "periods": sorted(flights["period"].astype(str).unique()),
    "memory_mb": round(flights.memory_usage(deep=True).sum() / 1024**2, 1),
    "load_seconds": round(time.perf_counter() - load_started, 1),
})

{'analysis_rows': 8989, 'periods': ['2021-12', '2022-03', '2022-06', '2022-09', '2022-12'], 'memory_mb': np.float64(1.7), 'load_seconds': 0.5}


In [5]:
# Compute each table once and reuse it throughout the narrative.
analysis_started = time.perf_counter()
analysis = export_business_analysis(flights, OUTPUT_ROOT, CONFIG)
print({
    "analysis_seconds": round(time.perf_counter() - analysis_started, 1),
    "output_root": str(analysis["output_root"]),
    "test_rows_read": 0,
})

{'analysis_seconds': 14.8, 'output_root': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\business_eda_smoke', 'test_rows_read': 0}


## 4. Executive network scorecard

OTP15 is the share of observed arrivals no more than 15 minutes late. Median
delay describes a typical flight; p90 and p95 reveal the operational tail that
drives disruption and customer impact.

In [6]:
kpis = analysis["kpis"]
display(kpis.to_frame("value").round(2))

,value
flights,8989.00
routes,3546.00
operators,206.00
departure_airports,550.00
arrival_airports,458.00
periods,5.00
arrival_observed,8989.00
arrival_otp15_pct,79.14
arrival_delay_median,1.95
arrival_delay_p90,26.95


## 5. Route demand and reliability

Raw percentages are not ranked without a volume rule. Wilson intervals express
uncertainty: small routes receive wider intervals, while high-volume routes are
estimated more precisely.

In [7]:
routes = analysis["routes"]
route_views = analysis["route_views"]

display(routes.head(20).round(2))
display(route_views["popular_reliable"].head(15).round(2))
display(route_views["least_reliable"].head(15).round(2))

,ADEP,ADES,route,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct
0,KJFK,EGLL,KJFK → EGLL,45,5,5,35,5.750000,1.980000,28.58,...,0,77.78,22.22,8.89,0.00,75.56,6,0.00,63.73,87.46
1,EGNX,EGAA,EGNX → EGAA,24,5,5,24,0.380000,0.400000,9.48,...,0,100.00,0.00,0.00,0.00,37.50,0,NaN,86.20,100.00
2,KLAX,EGLL,KLAX → EGLL,23,5,5,10,17.920000,21.379999,26.07,...,0,43.48,56.52,8.70,0.00,78.26,7,0.00,25.63,63.19
3,KEWR,EGLL,KEWR → EGLL,21,5,5,4,40.320000,28.320000,102.50,...,3,19.05,80.95,47.62,14.29,85.71,13,0.00,7.67,40.00
4,KJFK,LFPG,KJFK → LFPG,20,5,5,17,1.120000,-3.520000,25.89,...,0,85.00,15.00,10.00,0.00,55.00,3,0.00,63.96,94.76
5,OIIE,LTFM,OIIE → LTFM,20,5,5,19,-5.230000,-8.790000,10.70,...,0,95.00,5.00,0.00,0.00,10.00,2,100.00,76.39,99.11
6,KORD,EGLL,KORD → EGLL,18,5,5,6,29.440001,27.010000,74.57,...,3,33.33,66.67,44.44,16.67,88.89,9,0.00,16.28,56.25
7,CYUL,LFPG,CYUL → LFPG,18,5,5,2,34.500000,31.059999,54.12,...,1,11.11,88.89,50.00,5.56,66.67,14,7.14,3.10,32.80
8,KSFO,EGLL,KSFO → EGLL,17,5,5,6,22.410000,19.080000,43.87,...,0,35.29,64.71,23.53,0.00,82.35,6,16.67,17.31,58.70
9,RKSI,EDDF,RKSI → EDDF,17,5,5,9,12.200000,14.730000,27.37,...,0,52.94,47.06,5.88,0.00,52.94,6,16.67,30.96,73.83


,ADEP,ADES,route,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct


,ADEP,ADES,route,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct


In [8]:
# Volume and reliability answer different business questions, so both are shown.
display(plot_top_route_comparison(routes, CONFIG))
display(plot_route_volume_reliability(routes, float(kpis["arrival_otp15_pct"]), CONFIG))
plt.show()

<Figure size 1400x800 with 2 Axes>

<Figure size 1100x700 with 1 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_21892\172737208.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Business interpretation

- High volume + high OTP15: dependable core network.
- High volume + low OTP15: priority for operational intervention.
- Low volume + wide confidence interval: monitor before escalating.
- High p90 with acceptable median: usually reliable but exposed to severe tails.

Every route chart first excludes routes with fewer than two historical flights.
Executive reliability charts then apply the stronger 500-flight / three-period
rule. The two-flight rule prevents singleton routes from appearing as 0% or 100%
reliable while preserving broad volume charts.

The route table remains descriptive. It does not prove that a route causes a
delay because operator, airport, time and duration mix may differ.

## 6. Airline breadth and reliability

`AC Operator` is the operating carrier, not necessarily the marketing airline.
Route count measures network breadth; HHI measures concentration. A high HHI
means that a small number of routes dominate the operator's activity.

In [9]:
operators = analysis["operators"]
eligible_operators = operators.loc[
    operators["flights"].ge(CONFIG.executive_min_operator_flights)
]

display(
    eligible_operators[
        [
            "AC Operator", "flights", "routes", "airports",
            "arrival_otp15_pct", "arrival_delayed_30_pct",
            "arrival_delay_p90", "recovered_to_otp15_pct",
            "route_concentration_hhi",
        ]
    ].head(25).round(2)
)

,AC Operator,flights,routes,airports,arrival_otp15_pct,arrival_delayed_30_pct,arrival_delay_p90,recovered_to_otp15_pct,route_concentration_hhi


## 7. Which origin and destination airports are most problematic?

Origin and destination roles are analysed separately because they answer
different operational questions. Origin results reflect the conditions under
which a flight begins; destination results reflect the environment into which
the flight arrives. Airports require at least 30 observed flights in these smoke-
safe charts, and every estimate is accompanied by a 95% Wilson interval.

In [10]:
origin_airports = analysis["origin_airports"]
destination_airports = analysis["destination_airports"]

display(origin_airports.head(15).round(2))
display(destination_airports.head(15).round(2))

display(plot_airport_volume_reliability(
    origin_airports, "origin", float(kpis["arrival_otp15_pct"]), CONFIG
))
display(plot_airport_reliability_rankings(origin_airports, "origin", CONFIG))
display(plot_airport_volume_reliability(
    destination_airports, "destination", float(kpis["arrival_otp15_pct"]), CONFIG
))
display(plot_airport_reliability_rankings(destination_airports, "destination", CONFIG))
plt.show()

,role,airport,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,arrival_delay_p95,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct
0,origin,LTFM,342,5,5,310,0.550000,-0.45,14.08,23.14,...,0,90.64,9.36,2.05,0.00,59.94,24,25.00,87.09,93.29
1,origin,EDDP,335,5,5,326,-11.030000,-12.25,3.23,11.31,...,0,97.31,2.69,0.00,0.00,35.52,6,33.33,94.97,98.58
2,origin,EDDK,282,5,5,271,-6.040000,-6.10,9.68,13.77,...,0,96.10,3.90,0.35,0.00,35.46,6,0.00,93.15,97.81
3,origin,LFPG,247,5,5,198,3.290000,1.60,21.15,25.37,...,1,80.16,19.84,3.24,0.40,36.44,57,43.86,74.74,84.66
4,origin,KJFK,240,5,5,183,4.870000,-0.57,39.50,55.38,...,10,76.25,23.75,14.58,4.17,55.83,56,10.71,70.48,81.19
5,origin,OMDB,197,5,5,64,21.020000,19.43,39.63,48.58,...,2,32.49,67.51,18.27,1.02,70.05,100,4.00,26.34,39.31
6,origin,LLBG,147,5,5,127,3.090000,2.65,16.53,20.61,...,2,86.39,13.61,2.04,1.36,36.73,17,47.06,79.92,91.02
7,origin,EGNX,142,5,5,136,-0.740000,0.98,12.08,14.07,...,0,95.77,4.23,0.00,0.00,47.18,2,0.00,91.09,98.05
8,origin,EBBR,142,5,5,135,-0.990000,-1.36,12.42,14.82,...,0,95.07,4.93,0.00,0.00,32.39,7,42.86,90.17,97.59
9,origin,EBLG,138,5,5,124,-4.150000,-5.21,14.58,19.24,...,0,89.86,10.14,0.72,0.00,26.09,11,18.18,83.69,93.86


,role,airport,flights,periods_active,active_days,arrival_otp15_count,arrival_delay_mean,arrival_delay_median,arrival_delay_p90,arrival_delay_p95,...,arrival_delayed_60_count,arrival_otp15_pct,arrival_delayed_15_pct,arrival_delayed_30_pct,arrival_delayed_60_pct,worsened_pct,departed_delayed15,recovered_to_otp15_pct,arrival_otp15_ci_low_pct,arrival_otp15_ci_high_pct
0,destination,LTFM,436,5,5,317,6.180000,2.69,30.93,46.38,...,10,72.71,27.29,10.55,2.29,19.04,148,26.35,68.34,76.68
1,destination,EGLL,429,5,5,157,26.709999,22.07,60.62,81.82,...,45,36.60,63.40,32.40,10.49,85.78,189,2.65,32.18,41.26
2,destination,EHAM,406,5,5,274,14.410000,7.42,41.73,56.74,...,17,67.49,32.51,18.23,4.19,83.50,93,0.00,62.79,71.86
3,destination,LFPG,394,5,5,269,11.790000,8.36,36.04,46.21,...,10,68.27,31.73,12.44,2.54,66.50,99,11.11,63.52,72.67
4,destination,EDDF,383,5,5,320,0.920000,-2.43,22.21,37.62,...,8,83.55,16.45,6.01,2.09,16.19,87,36.78,79.51,86.93
5,destination,LTFJ,281,5,5,257,-1.940000,-3.67,13.92,22.12,...,1,91.46,8.54,3.56,0.36,23.84,36,52.78,87.61,94.19
6,destination,LEMD,249,5,5,178,8.830000,6.67,35.22,43.65,...,4,71.49,28.51,13.25,1.61,47.79,63,15.87,65.58,76.73
7,destination,EDDM,247,5,5,216,0.480000,-0.88,19.86,29.57,...,0,87.45,12.55,5.26,0.00,30.77,33,15.15,82.74,91.02
8,destination,LPPT,174,5,5,115,10.730000,8.67,28.21,35.00,...,3,66.09,33.91,8.62,1.72,64.37,43,4.65,58.78,72.71
9,destination,LIMC,137,5,5,123,-9.770000,-14.42,14.88,37.00,...,2,89.78,10.22,5.84,1.46,7.30,20,35.00,83.58,93.81


<Figure size 1100x700 with 2 Axes>

<Figure size 1400x800 with 2 Axes>

<Figure size 1100x700 with 2 Axes>

<Figure size 1400x800 with 2 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_21892\820635734.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Airport-chart interpretation

- The volume–reliability charts show scale, OTP15 and >30-minute exposure.
- The ranking charts use Wilson intervals, not raw percentages.
- A problematic origin is associated with weaker arrival outcomes for flights
  leaving that airport; a problematic destination is associated with weaker
  outcomes for flights arriving there.
- These are unadjusted associations. Route, operator, time and duration mix can
  explain part of the difference and should be controlled before assigning cause.

## 8. When is the network least reliable?

The heatmap uses scheduled departure time, which is known in advance. It helps
identify operational windows for staffing, disruption monitoring and customer
communications. It is descriptive and should not be interpreted as causal.

In [11]:
display(plot_time_reliability_heatmap(flights))
plt.show()

<Figure size 1400x600 with 2 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_21892\1303069179.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Delay propagation and recovery

Each hexagon groups many flights. The green diagonal represents equal departure
and arrival delay. Points below it recovered minutes; points above it worsened
after departure.

In [12]:
display(plot_departure_arrival_recovery(flights))
plt.show()

<Figure size 800x700 with 2 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_21892\3122058065.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Correlation: association, not causation

Pearson measures linear association. Spearman measures monotonic association and
is less sensitive to extreme delays. Post-event variables are valid for this
retrospective report but remain unavailable to the T−60 prediction model.

In [13]:
correlations = analysis["correlations"]
display(
    correlations.reindex(
        correlations["spearman_rho"].abs().sort_values(ascending=False).index
    ).head(20).round(4)
)

,variable_1,variable_2,rows,pearson_r,pearson_p,spearman_rho,spearman_p
17,schedule_buffer_min,recovery_minutes,8989,1.0000,0.0,1.0000,0.0
0,scheduled_duration_min,actual_duration_min,8989,0.9991,0.0,0.9951,0.0
8,actual_duration_min,Actual Distance Flown (nm),8989,0.9944,0.0,0.9923,0.0
2,scheduled_duration_min,Actual Distance Flown (nm),8989,0.9941,0.0,0.9889,0.0
25,Departure_Delay_Min,Arrival_Delay_Min,8989,0.8997,0.0,0.8362,0.0
16,schedule_buffer_min,Arrival_Delay_Min,8989,-0.4680,0.0,-0.5222,0.0
27,Arrival_Delay_Min,recovery_minutes,8989,-0.4680,0.0,-0.5222,0.0
20,Actual Distance Flown (nm),Arrival_Delay_Min,8989,0.4378,0.0,0.4576,0.0
11,actual_duration_min,Arrival_Delay_Min,8989,0.4357,0.0,0.4508,0.0
5,scheduled_duration_min,Arrival_Delay_Min,8989,0.4192,0.0,0.4143,0.0


## 11. Hypothesis tests: what exactly is being tested?

The catalog below separates tests already automated for the core report from
optional tests that require a business decision before execution.

**Core tests already implemented**

- **H01 — December change:** H0 says December 2021 and December 2022 have equal
  arrival OTP15. A two-proportion z-test is paired with the percentage-point
  difference.
- **H02 — En-route recovery:** H0 says median recovery equals zero among flights
  leaving more than 15 minutes late. Wilcoxon is used because delay differences
  are skewed and contain extreme events.
- **H03 — Haul bands:** H0 says all flight-duration bands share the same arrival-
  delay distribution. Kruskal–Wallis avoids a normality assumption.

**Recommended optional tests for selection**

- H04/H05: origin- and destination-airport association with OTP15.
- H06: operator association with OTP15, with a route-mix warning.
- H07: per-route stability across observed periods, corrected for multiple tests.
- H08: differences across scheduled departure-hour bands.
- H09: monotonic relationship between duration and delay.
- H10: adjusted operator effects after controlling for route, time and duration.

Large datasets can produce tiny p-values for unimportant differences. The report
therefore shows effect size and units alongside significance. Benjamini–Hochberg
is used only when many related hypotheses are tested simultaneously.

In [14]:
h0_catalog = analysis["hypothesis_test_catalog"]
hypothesis_tests = analysis["hypothesis_tests"]
display(h0_catalog)
display(hypothesis_tests.round(5))
display(plot_statistical_method_explainer())
plt.show()

,test_id,business_question,null_hypothesis,method,effect_size,recommendation,implementation
0,H01,Did December punctuality change between 2021 a...,December 2021 and December 2022 have equal arr...,Two-proportion z-test,Difference in percentage points,Core report,Automated
1,H02,Do flights leaving >15 minutes late recover ti...,Median en-route recovery equals zero minutes.,Paired Wilcoxon signed-rank test,Median minutes recovered,Core report,Automated
2,H03,Does the arrival-delay distribution differ by ...,All haul bands have the same arrival-delay dis...,Kruskal-Wallis test,Epsilon squared,Core report,Automated
3,H04,Is origin-airport punctuality associated with ...,Arrival OTP15 is independent of origin airport.,Chi-square test of independence,Cramér's V,Recommended optional,Awaiting selection
4,H05,Is destination-airport punctuality associated ...,Arrival OTP15 is independent of destination ai...,Chi-square test of independence,Cramér's V,Recommended optional,Awaiting selection
5,H06,Is punctuality associated with the operating c...,Arrival OTP15 is independent of AC Operator.,Chi-square test of independence,Cramér's V,Optional; route mix is a confounder,Awaiting selection
6,H07,Are route reliability rates stable across obse...,"For each eligible route, OTP15 is equal across...",Per-route chi-square tests + Benjamini-Hochberg,Maximum percentage-point change,Recommended optional,Awaiting selection
7,H08,Does punctuality differ across scheduled depar...,Arrival-delay distributions are equal across h...,Kruskal-Wallis test,Epsilon squared,Optional,Awaiting selection
8,H09,Is scheduled duration monotonically associated...,Spearman rho between duration and arrival dela...,Spearman rank-correlation test,Spearman rho,Optional; already reported in correlations,Available in correlation table
9,H10,Do operator differences remain after controlli...,Adjusted operator effects on P(delay >15) are ...,Adjusted logistic regression + likelihood-rati...,Adjusted odds ratios,Best for fair operator comparison,Future adjusted analysis


,test,test_id,null_hypothesis,statistic,p_value,effect,effect_unit,p_value_bh
0,December OTP15 equality,H01,December 2021 and December 2022 have equal OTP15,2.80623,0.00501,3.97315,percentage points (2021 minus 2022),0.00752
1,En-route recovery,H02,Median recovery is zero for flights departing ...,646182.00000,0.12023,-0.61667,median minutes recovered,0.12023
2,Delay equality across haul bands,H03,Arrival-delay distributions are equal across d...,1666.59216,0.00000,0.18515,Kruskal-Wallis epsilon squared,0.00000


<Figure size 1500x480 with 3 Axes>

C:\Users\celti\AppData\Local\Temp\ipykernel_21892\4132739489.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Generate report-ready assets

The export keeps tables, statistical outputs and figures separate. The future
Word report can therefore be regenerated without copying values manually.

In [15]:
print({
    "output_root": str(analysis["output_root"]),
    "tables": sorted(path.name for path in (OUTPUT_ROOT / "tables").glob("*.csv")),
    "figures": sorted(path.name for path in (OUTPUT_ROOT / "figures").glob("*.png")),
    "test_rows_read": 0,
})

{'output_root': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\business_eda_smoke', 'tables': ['destination_airport_performance.csv', 'least_reliable_routes.csv', 'network_kpis.csv', 'operator_performance.csv', 'origin_airport_performance.csv', 'popular_reliable_routes.csv', 'route_performance.csv', 'route_threshold_sensitivity.csv'], 'figures': ['departure_arrival_recovery.png', 'destination_airport_reliability_rankings.png', 'destination_airport_volume_reliability.png', 'origin_airport_reliability_rankings.png', 'origin_airport_volume_reliability.png', 'route_volume_reliability.png', 'statistical_method_explainer.png', 'time_reliability_heatmap.png', 'top_route_comparison.png'], 'test_rows_read': 0}


## 13. Reporting limitations

1. The development data contains five separated monthly snapshots, not a continuous year.
2. March 2023 is deliberately excluded to preserve the blind test.
3. Results describe operated flights; cancellations and diversions are absent.
4. Flight volume is not passenger, seat or revenue volume.
5. Operator comparisons are affected by route, airport and schedule mix.
6. `STATFOR Market Segment` loses detail in later files and must be interpreted cautiously.
7. Statistical significance does not establish operational causality.

Recommended next data: consecutive months, cancellations, aircraft capacity,
airport constraints, ATFM regulations and weather after the flight-only baseline
is fully documented.